# Book Recommendation System

**Implementation and evaluation using the Books, Ratings and Users datasets.**

This notebook implements popularity-based, content-based, collaborative-filtering and hybrid recommendation approaches. It evaluates rating prediction with **RMSE and MAE**, and top-N recommendation quality with **Precision@10, Recall@10, F1@10 and NDCG@10**.

**How to run:** Open in Google Colab, choose Runtime → Run all, and upload `Books.csv`, `Ratings.csv` and `Users.csv` when prompted. The notebook uses a fixed random seed for a reproducible train/test split. The ranking evaluation uses a deterministic sample of up to 300 eligible test users to keep runtime practical; the sample size is printed with the results.


In [1]:


import os
if not all(os.path.exists(f) for f in ["Books.csv", "Ratings.csv", "Users.csv"]):
    from google.colab import files
    uploaded = files.upload()


Saving Books.csv to Books.csv
Saving Ratings.csv to Ratings.csv
Saving Users.csv to Users.csv


In [ ]:
import pandas as pd

books = pd.read_csv("Books.csv", encoding="latin-1", low_memory=False, dtype={"ISBN": str})
ratings = pd.read_csv("Ratings.csv", dtype={"ISBN": str})
users = pd.read_csv("Users.csv")

print("Books:", books.shape)
print("Ratings:", ratings.shape)
print("Users:", users.shape)


Books: (271360, 8)
Ratings: (1149780, 3)
Users: (278858, 3)


In [ ]:

print("BOOKS")
print(books.columns.tolist())

print("\nRATINGS")
print(ratings.columns.tolist())

print("\nUSERS")
print(users.columns.tolist())


BOOKS
['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'Image-URL-S', 'Image-URL-M', 'Image-URL-L']

RATINGS
['User-ID', 'ISBN', 'Book-Rating']

USERS
['User-ID', 'Location', 'Age']


In [ ]:

print(books[['Book-Title', 'Book-Author']].head())

print("\nRatings:")
print(ratings.head())


                                          Book-Title           Book-Author
0                                Classical Mythology    Mark P. O. Morford
1                                       Clara Callan  Richard Bruce Wright
2                               Decision in Normandy          Carlo D'Este
3  Flu: The Story of the Great Influenza Pandemic...      Gina Bari Kolata
4                             The Mummies of Urumchi       E. J. W. Barber

Ratings:
   User-ID        ISBN  Book-Rating
0   276725  034545104X            0
1   276726  0155061224            5
2   276727  0446520802            0
3   276729  052165615X            3
4   276729  0521795028            6


In [ ]:

ratings = ratings[ratings['Book-Rating'] > 0]

print("Total ratings:", len(ratings))
print(ratings.head())


Total ratings: 433671
   User-ID        ISBN  Book-Rating
1   276726  0155061224            5
3   276729  052165615X            3
4   276729  0521795028            6
6   276736  3257224281            8
7   276737  0600570967            6


## 1. Popularity-Based Filtering

In [ ]:

popular_books = ratings.groupby('ISBN').agg(
    total_ratings=('Book-Rating', 'count'),
    average_rating=('Book-Rating', 'mean')
)

popular_books = popular_books[popular_books['total_ratings'] >= 100]

popular_books = popular_books.sort_values(
    'average_rating', ascending=False
)

print(popular_books.head(10))


            total_ratings  average_rating
ISBN                                     
0439139597            137        9.262774
0439136369            133        9.082707
0439136350            141        9.035461
043935806X            206        9.033981
0590353403            119        8.983193
0439139600            110        8.954545
0446310786            214        8.943925
059035342X            313        8.939297
0439064864            126        8.920635
0345339703            131        8.839695


In [ ]:

popular_books = popular_books.sort_values(
    'total_ratings', ascending=False
)

popular_books = popular_books.reset_index()

popular_books = popular_books.merge(books, on='ISBN')

print(popular_books[['Book-Title', 'Book-Author', 'total_ratings', 'average_rating']].head(10))


                                          Book-Title  ... average_rating
0                          The Lovely Bones: A Novel  ...       8.185290
1                                        Wild Animus  ...       4.390706
2                                  The Da Vinci Code  ...       8.435318
3                The Red Tent (Bestselling Backlist)  ...       8.182768
4    Divine Secrets of the Ya-Ya Sisterhood: A Novel  ...       7.887500
5  Harry Potter and the Sorcerer's Stone (Harry P...  ...       8.939297
6                            The Secret Life of Bees  ...       8.452769
7  Where the Heart Is (Oprah's Book Club (Paperba...  ...       8.142373
8                                    A Painted House  ...       7.338078
9                          Girl with a Pearl Earring  ...       7.982014

[10 rows x 4 columns]


## 2. Content-Based Filtering

In [ ]:

books = books.drop_duplicates('ISBN')

books['Book-Title'] = books['Book-Title'].fillna('')
books['Book-Author'] = books['Book-Author'].fillna('')

books['book_info'] = books['Book-Title'] + ' ' + books['Book-Author']

print(books[['Book-Title', 'Book-Author', 'book_info']].head())


                                          Book-Title  ...                                          book_info
0                                Classical Mythology  ...             Classical Mythology Mark P. O. Morford
1                                       Clara Callan  ...                  Clara Callan Richard Bruce Wright
2                               Decision in Normandy  ...                  Decision in Normandy Carlo D'Este
3  Flu: The Story of the Great Influenza Pandemic...  ...  Flu: The Story of the Great Influenza Pandemic...
4                             The Mummies of Urumchi  ...             The Mummies of Urumchi E. J. W. Barber

[5 rows x 3 columns]


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english', max_features=50000)

book_matrix = tfidf.fit_transform(books['book_info'])

print("Books processed:", book_matrix.shape[0])
print("Features created:", book_matrix.shape[1])


Books processed: 271360
Features created: 50000


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

book_similarity = cosine_similarity(
    book_matrix[0:1], book_matrix
)

print("Similarity calculated successfully!")
print("Total books compared:", book_similarity.shape[1])

Similarity calculated successfully!
Total books compared: 271360


In [ ]:
similarity_scores = book_similarity.flatten()

similar_book_indices = similarity_scores.argsort()[-11:][::-1]

print("Books similar to:", books.iloc[0]['Book-Title'])
print("\nRecommended Books:\n")

for i in similar_book_indices:
    if i != 0:
        print(books.iloc[i]['Book-Title'])

Books similar to: Classical Mythology

Recommended Books:

Classical Mythology
Classical mythology
Classical mythology
Who's Who in Classical Mythology (Who's Who Series)
Mythology and You : Classical Mythology and its Relevance in Today's World
Who's Who in Non-Classical Mythology
The Dictionary of Classical Mythology
Dictionary of Mythology: Mainly Classical
Dictionary of mythology, mainly classical
The Penguin Dictionary of Classical Mythology (Reference Books)


In [ ]:
def recommend_books(book_name):

    book_index = books[
        books['Book-Title'].str.lower() == book_name.lower()
    ].index

    if len(book_index) == 0:
        print("Book not found!")
        return

    index = book_index[0]

    similarity = cosine_similarity(
        book_matrix[index:index+1], book_matrix
    ).flatten()

    similar_indices = similarity.argsort()[::-1]

    print("Recommendations for:", book_name)
    print()

    count = 0
    for i in similar_indices:
        title = books.iloc[i]['Book-Title']

        if i != index and title.lower() != book_name.lower():
            print(title)
            count += 1

        if count == 10:
            break

In [ ]:
recommend_books("Classical Mythology")

Recommendations for: Classical Mythology

Who's Who in Classical Mythology (Who's Who Series)
Mythology and You : Classical Mythology and its Relevance in Today's World
Who's Who in Non-Classical Mythology
The Dictionary of Classical Mythology
Dictionary of Mythology: Mainly Classical
Dictionary of mythology, mainly classical
The Penguin Dictionary of Classical Mythology (Reference Books)
Clade
Spiked
Dracula (Classical Literature with Classical Music)


## 3. Collaborative Filtering

In [ ]:
ratings_count = ratings['User-ID'].value_counts()

active_users = ratings_count[ratings_count >= 50].index

filtered_ratings = ratings[
    ratings['User-ID'].isin(active_users)
]

print("Original ratings:", len(ratings))
print("Filtered ratings:", len(filtered_ratings))
print("Active users:", filtered_ratings['User-ID'].nunique())

Original ratings: 433671
Filtered ratings: 175023
Active users: 1332


In [ ]:
user_book_matrix = filtered_ratings.pivot_table(
    index='User-ID',
    columns='ISBN',
    values='Book-Rating'
)

print("Matrix shape:", user_book_matrix.shape)
print(user_book_matrix.head())

Matrix shape: (1332, 103253)
ISBN      0375404120   9022906116  ...  \8888809228\""  Ô½crosoft
User-ID                            ...                           
183              NaN          NaN  ...             NaN        NaN
254              NaN          NaN  ...             NaN        NaN
638              NaN          NaN  ...             NaN        NaN
643              NaN          NaN  ...             NaN        NaN
1025             NaN          NaN  ...             NaN        NaN

[5 rows x 103253 columns]


In [ ]:
user_book_matrix = user_book_matrix.fillna(0)

print("Missing values:", user_book_matrix.isnull().sum().sum())
print("Matrix shape:", user_book_matrix.shape)

Missing values: 0
Matrix shape: (1332, 103253)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_book_matrix)

print("Similarity calculated!")
print("Similarity matrix shape:", user_similarity.shape)

Similarity calculated!
Similarity matrix shape: (1332, 1332)


In [ ]:
user_id = user_book_matrix.index[0]

user_index = user_book_matrix.index.get_loc(user_id)

similar_users = user_similarity[user_index].argsort()[::-1]

print("Selected User ID:", user_id)
print("\nSimilar Users:\n")

count = 0

for i in similar_users:
    if i != user_index:
        print(user_book_matrix.index[i])
        count += 1

    if count == 5:
        break

Selected User ID: 183

Similar Users:

261379
83096
52350
80908
73681


In [ ]:
user_id = 183

user_index = user_book_matrix.index.get_loc(user_id)

similar_users = user_similarity[user_index].argsort()[::-1]

recommended_books = {}

for i in similar_users:
    if i == user_index:
        continue

    similar_user_id = user_book_matrix.index[i]
    similarity_score = user_similarity[user_index][i]

    if similarity_score <= 0:
        continue

    user_ratings = user_book_matrix.loc[similar_user_id]

    for isbn, rating in user_ratings.items():

        if rating >= 8 and user_book_matrix.loc[user_id, isbn] == 0:

            if isbn not in recommended_books:
                recommended_books[isbn] = 0

            recommended_books[isbn] += rating * similarity_score

top_recommendations = sorted(
    recommended_books,
    key=recommended_books.get,
    reverse=True
)[:10]

print("Recommendations for User:", user_id)
print()

for isbn in top_recommendations:
    book_name = books.loc[books['ISBN'] == isbn, 'Book-Title']

    if len(book_name) > 0:
        print(book_name.iloc[0])

Recommendations for User: 183

Harry Potter and the Goblet of Fire (Book 4)
NinguÃÂ©m morre sozinho: O adolescente e o suicÃÂ­dio (ColecÃÂ§ÃÂ£o universitÃÂ¡ria)
Harry Potter and the Prisoner of Azkaban (Book 3)
Harry Potter and the Sorcerer's Stone (Book 1)
Vozes e ruÃÂ­dos: DiÃÂ¡logos com adolescentes (Nosso mundo)
Inventem-se novos pais (Nosso mundo)


## 4. Evaluation — Train-Test Split and RMSE

In [ ]:
from sklearn.model_selection import train_test_split

train_ratings, test_ratings = train_test_split(
    filtered_ratings,
    test_size=0.2,
    random_state=42
)

print("Training ratings:", len(train_ratings))
print("Testing ratings:", len(test_ratings))

Training ratings: 140018
Testing ratings: 35005


In [ ]:
train_matrix = train_ratings.pivot_table(
    index='User-ID',
    columns='ISBN',
    values='Book-Rating'
)

print("Training matrix shape:", train_matrix.shape)

Training matrix shape: (1332, 86896)


In [ ]:
train_matrix = train_matrix.fillna(0)

from sklearn.metrics.pairwise import cosine_similarity

train_similarity = cosine_similarity(train_matrix)

print("Training similarity matrix:", train_similarity.shape)

Training similarity matrix: (1332, 1332)


In [ ]:
# Predict ratings for test interactions using user-user collaborative filtering.
# If a user or item is absent from the training matrix, fall back to the training global mean.
import numpy as np

global_mean = float(train_ratings['Book-Rating'].mean())
item_means = train_ratings.groupby('ISBN')['Book-Rating'].mean()
predicted_ratings = []

for _, row in test_ratings.iterrows():
    user_id = row['User-ID']
    isbn = row['ISBN']
    actual_rating = float(row['Book-Rating'])

    if user_id in train_matrix.index and isbn in train_matrix.columns:
        user_index = train_matrix.index.get_loc(user_id)
        item_user_ratings = train_matrix[isbn]
        similarities = train_similarity[user_index]
        rated_mask = item_user_ratings.to_numpy() > 0
        weighted_mask = (similarities > 0) & rated_mask

        if weighted_mask.any() and similarities[weighted_mask].sum() > 0:
            prediction = np.average(
                item_user_ratings.to_numpy()[weighted_mask],
                weights=similarities[weighted_mask]
            )
        else:
            prediction = float(item_means.get(isbn, global_mean))
    else:
        prediction = float(item_means.get(isbn, global_mean))

    prediction = float(np.clip(prediction, 1, 10))
    predicted_ratings.append((actual_rating, prediction))

print("Predictions generated:", len(predicted_ratings))


Predictions generated: 35005


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

actual = np.array([x[0] for x in predicted_ratings], dtype=float)
predicted = np.array([x[1] for x in predicted_ratings], dtype=float)

if len(actual) == 0:
    raise ValueError("No test ratings could be evaluated. Check the input datasets and train/test split.")

rmse = float(np.sqrt(mean_squared_error(actual, predicted)))
mae = float(mean_absolute_error(actual, predicted))

print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print("Ratings evaluated:", len(actual))


RMSE: 1.9721
MAE:  1.5430
Ratings evaluated: 35005


## 5. Final Comparison

In [ ]:
print("FINAL ALGORITHM COMPARISON")
print("--------------------------")

print("\n1. Popularity-Based")
print("Strength: Simple and recommends popular books.")
print("Weakness: Same recommendations for every user.")

print("\n2. Content-Based Filtering")
print("Strength: Recommends books with similar titles and authors.")
print("Weakness: Depends on book information and may miss user preferences.")

print("\n3. Collaborative Filtering")
print("Strength: Uses preferences of similar users.")
print("Weakness: Suffers from sparse ratings and cold-start problems.")

print("\nCollaborative Filtering RMSE:", round(rmse, 2))
print("Ratings evaluated:", len(actual))

FINAL ALGORITHM COMPARISON
--------------------------

1. Popularity-Based
Strength: Simple and recommends popular books.
Weakness: Same recommendations for every user.

2. Content-Based Filtering
Strength: Recommends books with similar titles and authors.
Weakness: Depends on book information and may miss user preferences.

3. Collaborative Filtering
Strength: Uses preferences of similar users.
Weakness: Suffers from sparse ratings and cold-start problems.

Collaborative Filtering RMSE: 1.97
Ratings evaluated: 35005


## 6. Hybrid Recommendation

This section combines the three approaches already implemented above. It does not replace the earlier code:
- Popularity score: books with stronger rating support and average ratings.
- Collaborative score: ratings from users similar to the selected user.
- Content score: similarity to books the selected user rated highly.

The scores are normalized before combining so that they are on a comparable scale.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Create a quick lookup from ISBN to the row in the content-based book matrix.
isbn_to_book_index = pd.Series(books.index, index=books['ISBN']).to_dict()

def recommend_hybrid(user_id, top_n=10):
    if user_id not in user_book_matrix.index:
        print("User not found in collaborative filtering data.")
        return pd.DataFrame()

    # Candidate books and popularity signal
    candidates = popular_books[['ISBN', 'Book-Title', 'Book-Author',
                                'total_ratings', 'average_rating']].copy()
    candidates = candidates.drop_duplicates('ISBN')

    # Popularity score: normalize rating count and average rating
    count_score = candidates['total_ratings'] / candidates['total_ratings'].max()
    rating_score = candidates['average_rating'] / 10
    candidates['popularity_score'] = 0.5 * count_score + 0.5 * rating_score

    # Collaborative score from the similar users calculated earlier
    user_index = user_book_matrix.index.get_loc(user_id)
    similarities = user_similarity[user_index].copy()
    similarities[user_index] = 0
    similar_indices = similarities.argsort()[::-1][:20]

    collaborative_scores = {}
    for i in similar_indices:
        similarity_score = similarities[i]
        if similarity_score <= 0:
            continue
        similar_user_ratings = user_book_matrix.iloc[i]
        for isbn, rating in similar_user_ratings.items():
            if rating >= 8 and user_book_matrix.loc[user_id, isbn] == 0:
                collaborative_scores[isbn] = collaborative_scores.get(isbn, 0) + rating * similarity_score

    candidates['collaborative_score'] = candidates['ISBN'].map(collaborative_scores).fillna(0)
    if candidates['collaborative_score'].max() > 0:
        candidates['collaborative_score'] /= candidates['collaborative_score'].max()

    # Content score based on this user's highly-rated books
    user_history = train_ratings[
        (train_ratings['User-ID'] == user_id) &
        (train_ratings['Book-Rating'] >= 8)
    ]['ISBN'].drop_duplicates().tolist()

    content_scores = {}
    valid_history = [isbn for isbn in user_history if isbn in isbn_to_book_index]
    if valid_history:
        history_indices = [isbn_to_book_index[isbn] for isbn in valid_history[:10]]
        user_profile = np.asarray(book_matrix[history_indices].mean(axis=0))
        similarities_to_profile = cosine_similarity(user_profile, book_matrix).flatten()
        content_scores = dict(zip(books['ISBN'], similarities_to_profile))

    candidates['content_score'] = candidates['ISBN'].map(content_scores).fillna(0)
    if candidates['content_score'].max() > 0:
        candidates['content_score'] /= candidates['content_score'].max()

    # Weighted hybrid score
    candidates['hybrid_score'] = (
        0.30 * candidates['popularity_score'] +
        0.40 * candidates['collaborative_score'] +
        0.30 * candidates['content_score']
    )

    # Remove books already rated by this user
    already_rated = set(user_book_matrix.loc[user_id][user_book_matrix.loc[user_id] > 0].index)
    candidates = candidates[~candidates['ISBN'].isin(already_rated)]

    return candidates.sort_values('hybrid_score', ascending=False).head(top_n)[
        ['ISBN', 'Book-Title', 'Book-Author', 'hybrid_score']
    ].reset_index(drop=True)

# Example: use a user already present in the collaborative filtering matrix
example_user_id = user_book_matrix.index[0]
print("Hybrid recommendations for User:", example_user_id)
display(recommend_hybrid(example_user_id, 10))

Hybrid recommendations for User: 183
         ISBN                                          Book-Title    Book-Author  hybrid_score
0  0439139600        Harry Potter and the Goblet of Fire (Book 4)  J. K. Rowling      0.557656
1  0439136350   Harry Potter and the Prisoner of Azkaban (Book 3)  J. K. Rowling      0.542827
2  0590353403      Harry Potter and the Sorcerer's Stone (Book 1)  J. K. Rowling      0.537376
3  0345342968                                      Fahrenheit 451   RAY BRADBURY      0.464216
4  043935806X  Harry Potter and the Order of the Phoenix (Book 5)  J. K. Rowling      0.382793
5  0439064864    Harry Potter and the Chamber of Secrets (Book 2)  J. K. Rowling      0.356965
6  0312924585                                Silence of the Lambs  Thomas Harris      0.352831
7  0156027321                                          Life of Pi    Yann Martel      0.345194
8  0385722206  Balzac and the Little Chinese Seamstress : A Novel      DAI SIJIE      0.324498
9  0316666343

## 7. Top-K Evaluation Metrics

RMSE in Section 4 evaluates rating prediction. Recommendation systems are also commonly evaluated by checking whether relevant books appear in the top K recommendations.

In this section:
- A rating of 8 or above is treated as a relevant book.
- Precision@K measures how many recommended books are relevant.
- Recall@K measures how many relevant test books were recommended.
- F1@K combines precision and recall.
- NDCG@K rewards relevant books appearing near the top of the list.

These metrics are calculated using the train/test split already created in Section 4.

In [ ]:
from collections import defaultdict
from functools import lru_cache
import numpy as np

# A rating >= 8 is treated as relevant positive feedback.
test_relevant_all = (
    test_ratings[test_ratings['Book-Rating'] >= 8]
    .groupby('User-ID')['ISBN']
    .apply(set)
    .to_dict()
)

# Use training data only to build recommendation candidates (avoids test-data leakage).
train_popularity = (
    train_ratings.groupby('ISBN')['Book-Rating']
    .agg(total_ratings='count', average_rating='mean')
    .reset_index()
)
train_popularity = train_popularity[train_popularity['total_ratings'] >= 2]
train_popularity = train_popularity.sort_values(
    ['average_rating', 'total_ratings'], ascending=False
)

# Deterministic evaluation sample: at most 300 users with at least one relevant test book.
evaluation_user_ids = sorted(test_relevant_all)[:300]
test_relevant = {uid: test_relevant_all[uid] for uid in evaluation_user_ids}
print(f"Eligible test users: {len(test_relevant_all)}")
print(f"Users evaluated (deterministic sample): {len(test_relevant)}")

# Cache each user's training history to avoid repeated dataframe scans.
train_seen = {
    uid: set(group['ISBN'])
    for uid, group in train_ratings.groupby('User-ID')
}
train_positive_history = {
    uid: group.loc[group['Book-Rating'] >= 8, 'ISBN'].drop_duplicates().tolist()
    for uid, group in train_ratings.groupby('User-ID')
}

popular_isbns = train_popularity['ISBN'].tolist()

def get_popular_recommendations(user_id, k=10):
    seen = train_seen.get(user_id, set())
    return [isbn for isbn in popular_isbns if isbn not in seen][:k]

# Convert once to NumPy so scoring is vectorised (the original Python loop over ~87k books per
# neighbour made this cell take far too long).
train_values = train_matrix.to_numpy()
train_isbns = train_matrix.columns.to_numpy()
train_isbn_pos = {isbn: pos for pos, isbn in enumerate(train_isbns)}

@lru_cache(maxsize=None)
def _collab_cached(user_id, k):
    if user_id not in train_matrix.index:
        return []
    user_idx = train_matrix.index.get_loc(user_id)
    sims = train_similarity[user_idx].copy()
    sims[user_idx] = 0
    neighbor_indices = np.argsort(sims)[::-1][:20]
    neighbor_indices = neighbor_indices[sims[neighbor_indices] > 0]
    if len(neighbor_indices) == 0:
        return []
    neighbor_ratings = train_values[neighbor_indices]
    liked = np.where(neighbor_ratings >= 8, neighbor_ratings, 0.0)
    scores = sims[neighbor_indices] @ liked          # sum(similarity * rating) per book
    for isbn in train_seen.get(user_id, set()):      # never recommend already-rated books
        pos = train_isbn_pos.get(isbn)
        if pos is not None:
            scores[pos] = 0
    top = np.argsort(scores)[::-1][:k]
    return [train_isbns[p] for p in top if scores[p] > 0]

def get_collaborative_recommendations(user_id, k=10):
    return _collab_cached(user_id, k)

@lru_cache(maxsize=None)
def get_content_recommendations(user_id, k=10):
    history = train_positive_history.get(user_id, [])
    valid_history = [isbn for isbn in history if isbn in isbn_to_book_index]
    if not valid_history:
        return get_popular_recommendations(user_id, k)
    query_indices = [isbn_to_book_index[isbn] for isbn in valid_history[:5]]
    user_profile = np.asarray(book_matrix[query_indices].mean(axis=0))
    scores = cosine_similarity(user_profile, book_matrix).ravel()
    seen = train_seen.get(user_id, set())
    ranked_indices = np.argsort(scores)[::-1][:k + len(seen) + 50]
    result = []
    for idx in ranked_indices:
        isbn = books.iloc[idx]['ISBN']
        if isbn not in seen:
            result.append(isbn)
        if len(result) >= k:
            break
    return result

def get_hybrid_recommendations_for_eval(user_id, k=10):
    # Weighted reciprocal-rank fusion of the three recommenders.
    recommendation_lists = [
        (0.30, get_popular_recommendations(user_id, 100)),
        (0.30, get_content_recommendations(user_id, 100)),
        (0.40, get_collaborative_recommendations(user_id, 100))
    ]
    combined_scores = defaultdict(float)
    for weight, items in recommendation_lists:
        for rank, isbn in enumerate(items, start=1):
            combined_scores[isbn] += weight / rank
    return [isbn for isbn, _ in sorted(
        combined_scores.items(), key=lambda x: x[1], reverse=True
    )[:k]]

def calculate_top_k_metrics(recommended, relevant, k=10):
    recommended = recommended[:k]
    relevant = set(relevant)
    hits = [1 if isbn in relevant else 0 for isbn in recommended]
    precision = sum(hits) / k
    recall = sum(hits) / len(relevant) if relevant else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    dcg = sum(hit / np.log2(rank + 2) for rank, hit in enumerate(hits))
    ideal_hits = min(len(relevant), k)
    idcg = sum(1 / np.log2(rank + 2) for rank in range(ideal_hits))
    ndcg = dcg / idcg if idcg else 0.0
    return precision, recall, f1, ndcg

def evaluate_model(recommendation_function, k=10):
    scores = []
    for user_id, relevant_books in test_relevant.items():
        recommendations = recommendation_function(user_id, k)
        scores.append(calculate_top_k_metrics(recommendations, relevant_books, k))
    return np.mean(scores, axis=0) if scores else np.zeros(4)

model_functions = {
    'Popularity-Based': get_popular_recommendations,
    'Content-Based': get_content_recommendations,
    'Collaborative Filtering': get_collaborative_recommendations,
    'Hybrid': get_hybrid_recommendations_for_eval
}

metric_rows = []
for model_name, model_function in model_functions.items():
    precision, recall, f1, ndcg = evaluate_model(model_function, k=10)
    metric_rows.append({
        'Model': model_name,
        'Precision@10': precision,
        'Recall@10': recall,
        'F1@10': f1,
        'NDCG@10': ndcg
    })

evaluation_results = pd.DataFrame(metric_rows).set_index('Model')
print("Top-K Evaluation Results (higher is better)")
display(evaluation_results.round(4))


Eligible test users: 1316
Users evaluated (deterministic sample): 300
Top-K Evaluation Results (higher is better)
                         Precision@10  Recall@10   F1@10  NDCG@10
Model                                                            
Popularity-Based               0.0000     0.0000  0.0000   0.0000
Content-Based                  0.0127     0.0123  0.0105   0.0168
Collaborative Filtering        0.0247     0.0175  0.0170   0.0362
Hybrid                         0.0233     0.0205  0.0178   0.0328


## 8. Updated Model Comparison

Use the measured metrics above to compare the four approaches. The highest score is not guaranteed to belong to the hybrid model; the result depends on the dataset split and model settings.

In [ ]:
print("MODEL COMPARISON")
print("----------------")
print("Popularity-Based: simple baseline; the same popular books may be shown to many users.")
print("Content-Based: recommends books with similar metadata; depends on metadata quality and user history.")
print("Collaborative Filtering: uses similar users' preferences; affected by sparse data and cold-start.")
print("Hybrid: combines popularity, content, and collaborative signals; more complex to compute.")

print("\nRating-prediction metrics (lower is better):")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"Number of rating predictions: {len(actual)}")

print("\nTop-N ranking metrics (higher is better):")
display(evaluation_results.round(4))

best_ndcg_model = evaluation_results['NDCG@10'].idxmax()
print(f"Best model by NDCG@10 on this evaluation sample: {best_ndcg_model}")
print("Note: results depend on the dataset, split, relevance threshold and sampled evaluation users.")


MODEL COMPARISON
----------------
Popularity-Based: simple baseline; the same popular books may be shown to many users.
Content-Based: recommends books with similar metadata; depends on metadata quality and user history.
Collaborative Filtering: uses similar users' preferences; affected by sparse data and cold-start.
Hybrid: combines popularity, content, and collaborative signals; more complex to compute.

Rating-prediction metrics (lower is better):
RMSE: 1.9721
MAE:  1.5430
Number of rating predictions: 35005

Top-N ranking metrics (higher is better):
                         Precision@10  Recall@10   F1@10  NDCG@10
Model                                                            
Popularity-Based               0.0000     0.0000  0.0000   0.0000
Content-Based                  0.0127     0.0123  0.0105   0.0168
Collaborative Filtering        0.0247     0.0175  0.0170   0.0362
Hybrid                         0.0233     0.0205  0.0178   0.0328
Best model by NDCG@10 on this evaluation sam